<a href="https://colab.research.google.com/github/aditya78i/PCA-2-DIP/blob/main/REAL_ESTATE_ADVISOR__PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Upload CSV & Install Libraries  (FIXED)        ║
# ╚══════════════════════════════════════════════════════════╝

# Install required libraries (zstandard handles the compressed file)
!pip install gradio scikit-learn plotly pandas numpy joblib zstandard --quiet

# Upload your compressed CSV file
from google.colab import files
import os, io

print('📂 Please upload your india_housing_prices compressed CSV...')
uploaded = files.upload()

RAW_PATH = list(uploaded.keys())[0]
print(f'\n✅ File uploaded: {RAW_PATH}  ({os.path.getsize(RAW_PATH)/1024/1024:.1f} MB compressed)')

# ── Decompress zstd file ───────────────────────────────────────────────────────
import zstandard as zstd
import pandas as pd

CSV_PATH = 'india_housing_prices.csv'

print('🔄 Decompressing zstd file...')
with open(RAW_PATH, 'rb') as f:
    raw_bytes = f.read()

# The file has an extra 4-byte prefix before the real zstd frame — skip it
dctx        = zstd.ZstdDecompressor()
decompressed = dctx.decompress(raw_bytes[4:], max_output_size=400_000_000)

with open(CSV_PATH, 'wb') as f:
    f.write(decompressed)

print(f'✅ Decompressed → {CSV_PATH}  ({len(decompressed)/1024/1024:.1f} MB)')

# ── Quick preview ──────────────────────────────────────────────────────────────
df_preview = pd.read_csv(CSV_PATH)
print(f'\nShape    : {df_preview.shape}')
print(f'Columns  : {df_preview.columns.tolist()}')
print(f'Missing  : {df_preview.isnull().sum().sum()}')
df_preview.head(3)

📂 Please upload your india_housing_prices compressed CSV...


Saving india_housing_prices (1)_compressed.csv to india_housing_prices (1)_compressed (1).csv

✅ File uploaded: india_housing_prices (1)_compressed (1).csv  (5.5 MB compressed)
🔄 Decompressing zstd file...
✅ Decompressed → india_housing_prices.csv  (39.2 MB)

Shape    : (250000, 23)
Columns  : ['ID', 'State', 'City', 'Locality', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status']
Missing  : 0


,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,...,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,...,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,...,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move


In [4]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Train Models on Your Real Data                 ║
# ╚══════════════════════════════════════════════════════════╝
import warnings, os, joblib
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.model_selection  import train_test_split
from sklearn.ensemble         import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics          import accuracy_score, f1_score, r2_score, classification_report

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f'✅ Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}')

# ── Feature engineering ───────────────────────────────────────────────────────
df['Amenities_Count'] = df['Amenities'].apply(lambda x: len(str(x).split(',')))

# ── Encode categorical columns ────────────────────────────────────────────────
CAT_COLS = [
    'State', 'City', 'Locality', 'Property_Type', 'Furnished_Status',
    'Public_Transport_Accessibility', 'Parking_Space', 'Security',
    'Amenities', 'Facing', 'Owner_Type', 'Availability_Status'
]

encoders = {}   # save encoders so Gradio app can reverse-map user inputs
le = LabelEncoder()
for col in CAT_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str)
        le_col = LabelEncoder()
        df[col] = le_col.fit_transform(df[col])
        encoders[col] = le_col

print('✅ Categorical columns encoded')
print('   Unique classes per column:')
for col, enc in encoders.items():
    print(f'     {col}: {list(enc.classes_)}')

# ── Define features ───────────────────────────────────────────────────────────
DROP  = ['ID', 'Good_Investment', 'Future_Price_5Y']
FEATS = [c for c in df.columns if c not in DROP]
print(f'\n✅ Feature columns ({len(FEATS)}): {FEATS}')

# ── Target variables ──────────────────────────────────────────────────────────
city_median = df.groupby('City')['Price_per_SqFt'].transform('median')

df['Good_Investment'] = (
    (df['Price_per_SqFt'] <= city_median) &
    (
        (df['BHK'] >= df['BHK'].median()) |
        (df['Amenities_Count'] >= df['Amenities_Count'].median()) |
        (df['Public_Transport_Accessibility'] >= df['Public_Transport_Accessibility'].median())
    )
).astype(int)

df['Future_Price_5Y'] = df['Price_in_Lakhs'] * (1.08 ** 5)

print(f'\n✅ Targets created')
print(f'   Good Investment: {df["Good_Investment"].value_counts().to_dict()}')

# ── Train/test split ──────────────────────────────────────────────────────────
X   = df[FEATS]
y_c = df['Good_Investment']
y_r = df['Future_Price_5Y']

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, y_c, test_size=0.2, random_state=42, stratify=y_c)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y_r, test_size=0.2, random_state=42)

print(f'\n✅ Split: {len(Xc_tr):,} train | {len(Xc_te):,} test')

# ── Train classifier ──────────────────────────────────────────────────────────
print('\n🚀 Training Random Forest Classifier...')
clf = RandomForestClassifier(
    n_estimators=150, max_depth=12, min_samples_split=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
clf.fit(Xc_tr, yc_tr)
y_pred_c = clf.predict(Xc_te)
print(f'   Accuracy : {accuracy_score(yc_te, y_pred_c):.4f}')
print(f'   F1 Score : {f1_score(yc_te, y_pred_c):.4f}')
print(classification_report(yc_te, y_pred_c, target_names=['Not Good','Good Investment']))

# ── Train regressor ───────────────────────────────────────────────────────────
print('🚀 Training Random Forest Regressor...')
reg = RandomForestRegressor(
    n_estimators=150, max_depth=12, min_samples_split=5,
    random_state=42, n_jobs=-1
)
reg.fit(Xr_tr, yr_tr)
y_pred_r = reg.predict(Xr_te)
print(f'   R² Score : {r2_score(yr_te, y_pred_r):.4f}')

# ── Feature importance ────────────────────────────────────────────────────────
fi = pd.Series(clf.feature_importances_, index=FEATS).sort_values(ascending=False)
print('\n📊 Top 10 Features:')
print(fi.head(10).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs('models', exist_ok=True)
joblib.dump(clf,      'models/clf.pkl')
joblib.dump(reg,      'models/reg.pkl')
joblib.dump(FEATS,    'models/features.pkl')
joblib.dump(encoders, 'models/encoders.pkl')
print('\n✅ All models saved to models/')

✅ Loaded 250,000 rows × 23 columns
Missing values: 0
✅ Categorical columns encoded
   Unique classes per column:
     State: ['Andhra Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Delhi', 'Gujarat', 'Haryana', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Odisha', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
     City: ['Ahmedabad', 'Amritsar', 'Bangalore', 'Bhopal', 'Bhubaneswar', 'Bilaspur', 'Chennai', 'Coimbatore', 'Cuttack', 'Dehradun', 'Durgapur', 'Dwarka', 'Faridabad', 'Gaya', 'Gurgaon', 'Guwahati', 'Haridwar', 'Hyderabad', 'Indore', 'Jaipur', 'Jamshedpur', 'Jodhpur', 'Kochi', 'Kolkata', 'Lucknow', 'Ludhiana', 'Mangalore', 'Mumbai', 'Mysore', 'Nagpur', 'New Delhi', 'Noida', 'Patna', 'Pune', 'Raipur', 'Ranchi', 'Silchar', 'Surat', 'Trivandrum', 'Vijayawada', 'Vishakhapatnam', 'Warangal']
     Locality: ['Locality_1', 'Locality_10', 'Locality_100', 'Locality_101', 'Locality_102', 'Locality_103', 'Locality_10

In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Launch Gradio Dashboard                        ║
# ╚══════════════════════════════════════════════════════════╝
import gradio as gr
import numpy  as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib, warnings
warnings.filterwarnings('ignore')

# ── Load artifacts ────────────────────────────────────────────────────────────
clf      = joblib.load('models/clf.pkl')
reg      = joblib.load('models/reg.pkl')
FEATS    = joblib.load('models/features.pkl')
encoders = joblib.load('models/encoders.pkl')

# ── Extract exact category options from your dataset ─────────────────────────
def choices(col): return list(encoders[col].classes_)

STATES       = choices('State')
CITIES       = choices('City')
PROP_TYPES   = choices('Property_Type')      # ['Apartment','Independent House','Villa']
FURNISHED    = choices('Furnished_Status')   # ['Furnished','Semi-furnished','Unfurnished']
TRANSPORT    = choices('Public_Transport_Accessibility')  # ['High','Low','Medium']
FACING       = choices('Facing')             # ['East','North','South','West']
OWNER        = choices('Owner_Type')         # ['Broker','Builder','Owner']
AVAIL        = choices('Availability_Status')# ['Ready_to_Move','Under_Construction']

# ── Encode helper ─────────────────────────────────────────────────────────────
def encode(col, val):
    try:
        return int(encoders[col].transform([str(val)])[0])
    except:
        return 0

# ── Build input row ───────────────────────────────────────────────────────────
def build_input(state, city, prop_type, bhk, size, price,
                year_built, floor_no, total_floors, furnished,
                parking, security, facing, owner, availability,
                transport, schools, hospitals, amenities_count):

    ppsf = (price * 100_000) / max(size, 1)
    age  = 2025 - year_built

    # Reconstruct an amenities string of the right length for encoding
    # (we use Amenities_Count as a numeric feature; Amenities col gets a median encode)
    row = {
        'State':                          encode('State', state),
        'City':                           encode('City', city),
        'Locality':                       encode('Locality', 'Locality_50'),
        'Property_Type':                  encode('Property_Type', prop_type),
        'BHK':                            int(bhk),
        'Size_in_SqFt':                   int(size),
        'Price_in_Lakhs':                 float(price),
        'Price_per_SqFt':                 round(ppsf, 6),
        'Year_Built':                     int(year_built),
        'Furnished_Status':               encode('Furnished_Status', furnished),
        'Floor_No':                       int(floor_no),
        'Total_Floors':                   int(total_floors),
        'Age_of_Property':                int(age),
        'Nearby_Schools':                 int(schools),
        'Nearby_Hospitals':               int(hospitals),
        'Public_Transport_Accessibility': encode('Public_Transport_Accessibility', transport),
        'Parking_Space':                  encode('Parking_Space', parking),
        'Security':                       encode('Security', security),
        'Amenities':                      0,   # median encode for unknown
        'Facing':                         encode('Facing', facing),
        'Owner_Type':                     encode('Owner_Type', owner),
        'Availability_Status':            encode('Availability_Status', availability),
        'Amenities_Count':                int(amenities_count),
    }

    df_in = pd.DataFrame([row])
    for c in FEATS:
        if c not in df_in.columns:
            df_in[c] = 0
    df_in = df_in[FEATS]
    return df_in, ppsf, age

# ── Main prediction + charts ──────────────────────────────────────────────────
def analyse(state, city, prop_type, bhk, size, price,
            year_built, floor_no, total_floors, furnished,
            parking, security, facing, owner, availability,
            transport, schools, hospitals, amenities_count):

    input_df, ppsf, age = build_input(
        state, city, prop_type, bhk, size, price,
        year_built, floor_no, total_floors, furnished,
        parking, security, facing, owner, availability,
        transport, schools, hospitals, amenities_count
    )

    proba     = clf.predict_proba(input_df)[0][1]
    pred      = int(clf.predict(input_df)[0])
    future_5y = float(reg.predict(input_df)[0])

    roi   = ((future_5y - price) / price) * 100 if price > 0 else 0
    cagr  = ((future_5y / price) ** 0.2 - 1) * 100 if price > 0 else 0
    score = int(min(proba * 50 + min(roi / 60 * 50, 50), 100))

    good   = pred == 1
    color  = '#10b981' if good else '#ef4444'
    emoji  = '✅' if good else '⚠️'
    label  = 'GOOD INVESTMENT' if good else 'NOT RECOMMENDED'
    bg     = 'linear-gradient(135deg,#064e3b,#065f46)' if good else 'linear-gradient(135deg,#7f1d1d,#991b1b)'
    roi_c  = '#4ade80' if roi > 0 else '#f87171'
    cagr_c = '#4ade80' if cagr >= 7 else '#fbbf24'

    # ── Verdict HTML ──────────────────────────────────────────────────────────
    verdict = f"""
    <div style='background:{bg};border:2px solid {color};border-radius:14px;
                padding:22px 28px;text-align:center;font-family:Inter,sans-serif;
                box-shadow:0 6px 24px rgba(0,0,0,.3);'>
      <div style='font-size:2rem;font-weight:800;color:{color};margin-bottom:8px;'>
        {emoji} {label}
      </div>
      <div style='color:#cbd5e1;font-size:.95rem;'>
        Confidence <b style='color:#fff'>{proba:.0%}</b>
        &nbsp;·&nbsp; Score <b style='color:#fff'>{score}/100</b>
      </div>
      <div style='margin-top:16px;display:flex;justify-content:center;
                  gap:30px;flex-wrap:wrap;'>
        {''.join([
            f"<div style='text-align:center;'>"
            f"<div style='color:#94a3b8;font-size:.7rem;text-transform:uppercase;letter-spacing:1px;'>{lbl}</div>"
            f"<div style='color:{clr};font-size:1.25rem;font-weight:700;'>{val}</div></div>"
            for lbl, val, clr in [
                ('Current Price',  f'₹{price:,.0f} L',     '#f1f5f9'),
                ('5-Year Value',   f'₹{future_5y:,.1f} L', '#f1f5f9'),
                ('Total ROI',      f'{roi:.1f}%',           roi_c),
                ('CAGR',           f'{cagr:.1f}%',          cagr_c),
                ('Price / SqFt',   f'₹{ppsf:,.0f}',        '#f1f5f9'),
                ('Property Age',   f'{age} yrs',            '#f1f5f9'),
            ]
        ])}
      </div>
    </div>"""

    # ── Chart 1: Price projection + Score gauge ───────────────────────────────
    years  = list(range(0, 11))
    prices = [price * (1.08 ** y) for y in years]

    fig1 = make_subplots(
        rows=1, cols=2,
        subplot_titles=('📈 10-Year Price Projection (8% p.a.)', '🎯 Investment Score'),
        specs=[[{'type': 'xy'}, {'type': 'indicator'}]]
    )
    fig1.add_trace(go.Scatter(
        x=years, y=prices, mode='lines+markers+text',
        text=[f'₹{p:.0f}L' if i % 2 == 0 else '' for i, p in enumerate(prices)],
        textposition='top center', textfont=dict(size=9, color='#94a3b8'),
        line=dict(color='#10b981', width=3),
        marker=dict(size=7, color='#34d399'),
        fill='tozeroy', fillcolor='rgba(16,185,129,0.08)',
        name='Projected Value'
    ), row=1, col=1)
    fig1.add_trace(go.Scatter(
        x=[0, 10], y=[price, price], mode='lines',
        line=dict(color='#64748b', dash='dash', width=1.5),
        name='Purchase Price'
    ), row=1, col=1)
    fig1.add_trace(go.Indicator(
        mode='gauge+number',
        value=score,
        number={'suffix': '/100', 'font': {'size': 40, 'color': color}},
        gauge={
            'axis': {'range': [0, 100], 'tickcolor': '#94a3b8'},
            'bar':  {'color': color, 'thickness': 0.28},
            'bgcolor': '#1e293b', 'bordercolor': '#334155',
            'steps': [
                {'range': [0,  40],  'color': 'rgba(239,68,68,0.15)'},
                {'range': [40, 65],  'color': 'rgba(251,191,36,0.15)'},
                {'range': [65, 100], 'color': 'rgba(16,185,129,0.15)'},
            ],
        }
    ), row=1, col=2)
    fig1.update_layout(
        height=340, paper_bgcolor='#0f172a', plot_bgcolor='#1e293b',
        font=dict(color='#cbd5e1'),
        legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color='#94a3b8')),
        margin=dict(l=20, r=20, t=50, b=20)
    )
    fig1.update_xaxes(title_text='Year', gridcolor='#334155', row=1, col=1)
    fig1.update_yaxes(title_text='₹ Lakhs', gridcolor='#334155', row=1, col=1)

    # ── Chart 2: Yearly gains + Property radar ────────────────────────────────
    yr_labels = [f'Y{y}' for y in range(1, 11)]
    gains     = [round(price * (1.08 ** y) - price, 1) for y in range(1, 11)]
    bar_cols  = ['#10b981' if g > 0 else '#ef4444' for g in gains]

    # Radar scores (0–10)
    ppsf_sc   = max(0, min(10, 10 - (ppsf - 3000) / 700))
    trans_sc  = {'Low': 2, 'Medium': 6, 'High': 10}.get(transport, 5)
    amen_sc   = min(amenities_count * 0.8, 10)
    age_sc    = max(0, 10 - age * 0.28)
    bhk_sc    = min(bhk * 2, 10)
    radar_v   = [ppsf_sc, trans_sc, amen_sc, age_sc, bhk_sc]
    radar_l   = ['Value/Money', 'Transport', 'Amenities', 'Prop. Age', 'BHK']

    fig2 = make_subplots(
        rows=1, cols=2,
        subplot_titles=('💰 Cumulative Gain Over 10 Years (₹ L)', '📊 Property Profile Score'),
        specs=[[{'type': 'xy'}, {'type': 'polar'}]]
    )
    fig2.add_trace(go.Bar(
        x=yr_labels, y=gains, marker_color=bar_cols,
        text=[f'₹{g}L' for g in gains], textposition='outside',
        textfont=dict(size=9, color='#94a3b8'), name='Gain'
    ), row=1, col=1)
    fig2.add_trace(go.Scatterpolar(
        r=radar_v + [radar_v[0]],
        theta=radar_l + [radar_l[0]],
        fill='toself', fillcolor='rgba(56,189,248,0.2)',
        line=dict(color='#38bdf8', width=2.5),
        name='Profile'
    ), row=1, col=2)
    fig2.update_layout(
        height=360, paper_bgcolor='#0f172a', plot_bgcolor='#1e293b',
        font=dict(color='#cbd5e1'),
        polar=dict(
            bgcolor='#1e293b',
            radialaxis=dict(visible=True, range=[0, 10],
                            color='#64748b', gridcolor='#334155'),
            angularaxis=dict(color='#94a3b8', gridcolor='#334155')
        ),
        margin=dict(l=20, r=20, t=50, b=20),
        showlegend=False
    )
    fig2.update_xaxes(gridcolor='#334155', row=1, col=1)
    fig2.update_yaxes(title_text='₹ Lakhs', gridcolor='#334155', row=1, col=1)

    # ── Chart 3: Confidence gauge ─────────────────────────────────────────────
    fig3 = go.Figure(go.Indicator(
        mode='gauge+number+delta',
        value=round(proba * 100, 1),
        delta={'reference': 50,
               'increasing': {'color': '#10b981'},
               'decreasing': {'color': '#ef4444'}},
        number={'suffix': '%', 'font': {'size': 44, 'color': color}},
        title={'text': '🤖 Model Confidence', 'font': {'color': '#e2e8f0', 'size': 14}},
        gauge={
            'axis': {'range': [0, 100], 'tickcolor': '#94a3b8'},
            'bar':  {'color': color, 'thickness': 0.3},
            'bgcolor': '#1e293b', 'bordercolor': '#334155',
            'steps': [
                {'range': [0,  50],  'color': 'rgba(239,68,68,0.12)'},
                {'range': [50, 75],  'color': 'rgba(251,191,36,0.12)'},
                {'range': [75, 100], 'color': 'rgba(16,185,129,0.12)'},
            ]
        }
    ))
    fig3.update_layout(
        height=280, paper_bgcolor='#0f172a',
        font=dict(color='#cbd5e1'),
        margin=dict(l=30, r=30, t=40, b=10)
    )

    # ── Advisor insights ──────────────────────────────────────────────────────
    tips = []
    # Price check
    if ppsf > 8000:
        tips.append(('#ef4444', '🔴', 'High price/SqFt — check comparable listings nearby'))
    elif ppsf > 5000:
        tips.append(('#f59e0b', '🟡', 'Moderate price/SqFt — within city average range'))
    else:
        tips.append(('#10b981', '🟢', 'Competitive price/SqFt — good value for money'))
    # Age check
    if age > 25:
        tips.append(('#ef4444', '🔴', f'Property is {age} years old — factor in renovation costs'))
    elif age <= 5:
        tips.append(('#10b981', '🟢', 'Near-new construction — minimal maintenance expected'))
    else:
        tips.append(('#f59e0b', '🟡', f'{age}-year-old property — get a structural inspection done'))
    # Amenities
    if amenities_count >= 5:
        tips.append(('#10b981', '🟢', f'{amenities_count} amenities — boosts rental yield and resale value'))
    elif amenities_count <= 1:
        tips.append(('#ef4444', '🔴', 'Very few amenities — may reduce tenant and buyer demand'))
    else:
        tips.append(('#f59e0b', '🟡', f'{amenities_count} amenities — moderate; room to negotiate price'))
    # Transport
    if transport == 'High':
        tips.append(('#10b981', '🟢', 'High transport accessibility — strong appreciation driver'))
    elif transport == 'Low':
        tips.append(('#ef4444', '🔴', 'Low transport access — may limit future buyer pool'))
    else:
        tips.append(('#f59e0b', '🟡', 'Medium transport access — acceptable for most buyers'))
    # BHK
    if bhk >= 3:
        tips.append(('#10b981', '🟢', f'{bhk} BHK — family-sized units have strong long-term demand'))
    elif bhk == 1:
        tips.append(('#f59e0b', '🟡', '1 BHK — high rental demand but lower resale value'))
    # Availability
    if availability == 'Ready_to_Move':
        tips.append(('#10b981', '🟢', 'Ready to move — no construction risk, immediate rental income possible'))
    else:
        tips.append(('#f59e0b', '🟡', 'Under construction — lower price but higher completion risk'))
    # ROI
    if roi < 20:
        tips.append(('#ef4444', '🔴', f'5Y ROI of {roi:.1f}% is below the 40% market benchmark'))
    elif roi >= 40:
        tips.append(('#10b981', '🟢', f'Strong 5Y ROI of {roi:.1f}% — well above market average'))

    rows_html = ''.join([
        f"""<div style='display:flex;align-items:center;gap:10px;
            background:#1e293b;border-left:4px solid {col};
            border-radius:0 8px 8px 0;padding:10px 14px;
            margin-bottom:7px;font-family:Inter,sans-serif;'>
          <span style='font-size:1rem;'>{icon}</span>
          <span style='color:#cbd5e1;font-size:.9rem;'>{text}</span>
        </div>"""
        for col, icon, text in tips[:7]
    ])
    insights = f"""
    <div style='font-family:Inter,sans-serif;'>
      <div style='color:#38bdf8;font-weight:600;font-size:1rem;
                  border-bottom:1px solid #334155;padding-bottom:8px;
                  margin-bottom:12px;'>💡 Advisor Insights</div>
      {rows_html}
    </div>"""

    # ── Summary table ─────────────────────────────────────────────────────────
    summary = pd.DataFrame({
        'Metric': ['Decision','Confidence','Score','Current Price',
                   '5-Year Projected Value','Total ROI (5Y)',
                   'CAGR','Price per SqFt','Property Age','Amenities'],
        'Value':  [
            f'{emoji} {label}',
            f'{proba:.1%}',
            f'{score} / 100',
            f'₹ {price:,} Lakhs',
            f'₹ {future_5y:,.1f} Lakhs',
            f'{roi:.1f} %',
            f'{cagr:.2f} % per annum',
            f'₹ {ppsf:,.0f} per SqFt',
            f'{age} years',
            str(int(amenities_count)),
        ]
    })

    return verdict, fig1, fig2, fig3, insights, summary


# ── Gradio UI ─────────────────────────────────────────────────────────────────
theme = gr.themes.Base(
    primary_hue='sky',
    neutral_hue='slate',
    font=gr.themes.GoogleFont('Inter')
).set(
    body_background_fill='#0f172a',
    block_background_fill='#1e293b',
    block_border_color='#334155',
    block_label_text_color='#94a3b8',
    input_background_fill='#0f172a',
    input_border_color='#334155',
    button_primary_background_fill='#0ea5e9',
    button_primary_text_color='#ffffff',
)

with gr.Blocks(theme=theme, title='🏠 Real Estate Investment Advisor') as demo:

    gr.HTML("""
    <div style='background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);
                padding:24px 32px;border-radius:14px;margin-bottom:12px;
                box-shadow:0 8px 32px rgba(0,0,0,.4);'>
      <h1 style='color:#fff;margin:0;font-size:1.9rem;'>🏠 Real Estate Investment Advisor</h1>
      <p style='color:#94a3b8;margin:6px 0 0;font-size:.9rem;'>
        Trained on <b style='color:#38bdf8'>250,000 real Indian property listings</b>
        · Random Forest ML · Predict investment potential & 5-year value
      </p>
    </div>
    """)

    with gr.Row():
        # ── Input panel ───────────────────────────────────────────────────────
        with gr.Column(scale=1, min_width=320):
            gr.Markdown('### 🏗️ Property Details')

            with gr.Group():
                gr.Markdown('**📍 Location**')
                with gr.Row():
                    i_state = gr.Dropdown(STATES,     value=STATES[0],    label='State')
                    i_city  = gr.Dropdown(CITIES,     value=CITIES[0],    label='City')

            with gr.Group():
                gr.Markdown('**🏠 Property Basics**')
                with gr.Row():
                    i_prop  = gr.Dropdown(PROP_TYPES, value=PROP_TYPES[0],label='Property Type')
                    i_bhk   = gr.Slider(1, 5, value=2, step=1,            label='BHK')
                with gr.Row():
                    i_size  = gr.Slider(500,  5000, value=1500, step=50,  label='Size (SqFt)')
                    i_price = gr.Slider(10,   500,  value=120,  step=5,   label='Price (₹ Lakhs)')
                with gr.Row():
                    i_year  = gr.Slider(1990, 2023, value=2012, step=1,   label='Year Built')
                with gr.Row():
                    i_floor = gr.Slider(0,    30,   value=5,    step=1,   label='Floor No.')
                    i_tflr  = gr.Slider(1,    30,   value=10,   step=1,   label='Total Floors')

            with gr.Group():
                gr.Markdown('**✨ Features**')
                with gr.Row():
                    i_furn  = gr.Dropdown(FURNISHED, value=FURNISHED[0],  label='Furnished Status')
                    i_face  = gr.Dropdown(FACING,    value=FACING[0],     label='Facing')
                with gr.Row():
                    i_park  = gr.Radio(['Yes','No'], value='Yes',          label='Parking Space')
                    i_sec   = gr.Radio(['Yes','No'], value='Yes',          label='Security')
                with gr.Row():
                    i_own   = gr.Dropdown(OWNER,     value=OWNER[0],      label='Owner Type')
                    i_avail = gr.Dropdown(AVAIL,     value=AVAIL[0],      label='Availability')

            with gr.Group():
                gr.Markdown('**📍 Accessibility & Amenities**')
                i_trans   = gr.Dropdown(TRANSPORT, value='Medium',        label='Transport Accessibility')
                with gr.Row():
                    i_sch   = gr.Slider(1, 10, value=5, step=1,           label='Nearby Schools')
                    i_hosp  = gr.Slider(1, 10, value=4, step=1,           label='Nearby Hospitals')
                i_amen    = gr.Slider(0, 8,  value=3, step=1,
                                      label='Amenities Count',
                                      info='e.g. 3 = Clubhouse, Garden, Gym')

            btn = gr.Button('🔍  Analyse Investment', variant='primary', size='lg')

        # ── Output panel ──────────────────────────────────────────────────────
      # ── Output panel ──────────────────────────────────────────────────────
        with gr.Column(scale=2):
            o_verdict  = gr.HTML(label='Verdict')
            o_chart1   = gr.Plot(label='Price Projection & Score')
            o_chart2   = gr.Plot(label='Gains & Property Profile')
            with gr.Row():
                with gr.Column(scale=1):
                    o_chart3 = gr.Plot(label='Confidence')
                with gr.Column(scale=2):
                    o_insights = gr.HTML(label='Insights')
            o_summary  = gr.Dataframe(label='📋 Full Summary', interactive=False)

    inputs  = [i_state, i_city, i_prop, i_bhk, i_size, i_price,
                i_year, i_floor, i_tflr, i_furn,
                i_park, i_sec, i_face, i_own, i_avail,
                i_trans, i_sch, i_hosp, i_amen]
    outputs = [o_verdict, o_chart1, o_chart2, o_chart3, o_insights, o_summary]

    btn.click(fn=analyse, inputs=inputs, outputs=outputs)

    gr.HTML("""
    <div style='text-align:center;color:#475569;font-size:.8rem;padding:16px 0 4px;'>
      Real Estate Investment Advisor · Random Forest ML ·
      Trained on 250,000 Indian Property Listings
    </div>""")

# share=True gives a public link valid for 72 hours
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d23a1c717120526d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
